In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.linear_model import (
    LinearRegression, RANSACRegressor, 
    HuberRegressor, TheilSenRegressor
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    RobustScaler, OneHotEncoder, 
    LabelEncoder, StandardScaler
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from category_encoders import TargetEncoder
import warnings
warnings.filterwarnings('ignore')

class MeanEncoder:
    """
    Custom mean encoder that handles mixed types within columns
    """
    def __init__(self, handle_unknown='value'):
        self.handle_unknown = handle_unknown
        self.encodings_ = {}
        self.global_mean_ = None
        
    def _convert_to_string(self, x):
        """Convert any value to string representation safely"""
        if pd.isna(x):
            return 'MISSING'
        return str(x)
    
    def get_feature_names_out(self, feature_names=None):
        """Return feature names for output features."""
        if feature_names is None:
            # If no feature names provided, create generic ones
            return np.array([f"mean_encoded_{i}" for i in range(self.n_features_)])
        else:
            # Create encoded feature name for each input feature
            return np.array([f"{fname}_mean_encoded" for fname in feature_names])

    def fit(self, X, y):

        self.n_features_ = X.shape[1] if isinstance(X, np.ndarray) else 1

        if y is None:
            raise ValueError("y cannot be None in fit!")
            
        # Handle different input types and convert to string
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0].apply(self._convert_to_string)
        elif isinstance(X, np.ndarray):
            if X.ndim > 1:
                X = X[:, 0]
            X = pd.Series(X).apply(self._convert_to_string)
        else:
            X = pd.Series(X).apply(self._convert_to_string)
            
        # Calculate global mean for unknown categories
        self.global_mean_ = np.mean(y)
        
        # Calculate mean target value for each category
        categories = pd.Series(X)
        targets = pd.Series(y)
        self.encodings_ = targets.groupby(categories).mean().to_dict()
        
        return self
        
    def transform(self, X):
        if not self.encodings_:
            raise ValueError("Encoder not fitted. Call fit before transform.")
            
        # Handle different input types and convert to string
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0].apply(self._convert_to_string)
        elif isinstance(X, np.ndarray):
            if X.ndim > 1:
                X = X[:, 0]
            X = pd.Series(X).apply(self._convert_to_string)
        else:
            X = pd.Series(X).apply(self._convert_to_string)
            
        # Transform using dictionary mapping with fallback to global mean
        result = np.array([self.encodings_.get(x, self.global_mean_) for x in X])
        return result.reshape(-1, 1)
        
    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

class MixedTypeRobustEstimator:
    def __init__(
        self,
        categorical_cols=None,
        boolean_cols=None,
        numeric_cols=None,
        contamination=0.1,
        n_iterations=100,
        cv_folds=5,
        random_state=42,
        categorical_encoding='auto',
        feature_selection_threshold=0.01
    ):
        # Initialize all attributes explicitly
        self.categorical_cols = categorical_cols if categorical_cols is not None else []
        self.boolean_cols = boolean_cols if boolean_cols is not None else []
        self.numeric_cols = numeric_cols if numeric_cols is not None else []
        self.contamination = contamination
        self.n_iterations = n_iterations
        self.cv_folds = cv_folds
        self.random_state = random_state
        self.categorical_encoding = categorical_encoding
        self.feature_threshold = feature_selection_threshold
        self.preprocessor_ = None
        self.final_estimator_ = None
        self.feature_names_ = None
        self.feature_importances_ = None
        
    def _preprocess_column(self, X, column):
        """
        Preprocess a column to handle mixed types
        """
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        # If the column is numeric (or should be), try to convert all values to float
        if column in self.numeric_cols:
            try:
                return pd.to_numeric(X[column], errors='coerce')  # Convert to numeric, set invalid to NaN
            except:
                return X[column]
                
        # For categorical columns, convert everything to string
        if column in self.categorical_cols:
            return X[column].apply(lambda x: str(x) if pd.notnull(x) else x)
                
        return X[column]
    
    def _infer_column_types(self, X):
        """
        Automatically infer column types if not provided
        """
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        # Only infer if no columns were explicitly provided
        if not any([self.categorical_cols, self.boolean_cols, self.numeric_cols]):
            self.categorical_cols = []
            self.boolean_cols = []
            self.numeric_cols = []
            
            for col in X.columns:
                # Get non-null values for better type inference
                non_null_values = X[col].dropna()
                
                # Skip empty columns
                if len(non_null_values) == 0:
                    self.numeric_cols.append(col)  # Default to numeric
                    continue
                    
                # Check types of non-null values
                if pd.api.types.is_bool_dtype(non_null_values):
                    self.boolean_cols.append(col)
                elif pd.api.types.is_numeric_dtype(non_null_values):
                    unique_vals = len(non_null_values.unique())
                    if unique_vals < 10 and unique_vals < len(X) * 0.05:
                        self.categorical_cols.append(col)
                    else:
                        self.numeric_cols.append(col)
                elif pd.api.types.is_object_dtype(non_null_values) or pd.api.types.is_categorical_dtype(non_null_values):
                    # Try to convert to numeric if possible
                    try:
                        numeric_values = pd.to_numeric(non_null_values)
                        # If successful, check if it should be categorical or numeric
                        unique_vals = len(numeric_values.unique())
                        if unique_vals < 10 and unique_vals < len(X) * 0.05:
                            self.categorical_cols.append(col)
                        else:
                            self.numeric_cols.append(col)
                    except (ValueError, TypeError):
                        self.categorical_cols.append(col)
                else:
                    # Try to infer numeric even for mixed types
                    try:
                        pd.to_numeric(non_null_values)
                        self.numeric_cols.append(col)
                    except (ValueError, TypeError):
                        self.categorical_cols.append(col)

            print("\nColumn type inference:")
            print("Numeric columns:", self.numeric_cols)
            print("Boolean columns:", self.boolean_cols)
            print("Categorical columns:", self.categorical_cols)
    
    def _create_preprocessing_pipeline(self, X, y=None):
        """
        Create preprocessing pipeline for different column types
        """
        transformers = []
        
        # Numeric features pipeline
        if self.numeric_cols:
            numeric_pipe = Pipeline([
                ('imputer', SimpleImputer(strategy='mean')),
                ('scaler', RobustScaler())
            ])
            transformers.append(('numeric', numeric_pipe, self.numeric_cols))
        
        # Boolean features pipeline
        if self.boolean_cols:
            boolean_pipe = Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('scaler', StandardScaler())
            ])
            transformers.append(('boolean', boolean_pipe, self.boolean_cols))
        
        # Categorical features pipeline
        if self.categorical_cols:
            # Preprocess categorical columns - convert to string first
            X = X.copy()
            for col in self.categorical_cols:
                X[col] = X[col].apply(lambda x: str(x) if pd.notnull(x) else x)
            
            if self.categorical_encoding == 'auto':
                # Split based on cardinality
                high_card_cols = []
                low_card_cols = []
                
                for col in self.categorical_cols:
                    n_unique = len(pd.Series(X[col]).dropna().unique())
                    if n_unique > 10:
                        high_card_cols.append(col)
                    else:
                        low_card_cols.append(col)
                
                if high_card_cols:
                    mean_pipe = Pipeline([
                        ('imputer', SimpleImputer(strategy='most_frequent')),  # Changed from constant
                        ('encoder', MeanEncoder())
                    ])
                    transformers.append(('categorical_high', mean_pipe, high_card_cols))
                
                if low_card_cols:
                    onehot_pipe = Pipeline([
                        ('imputer', SimpleImputer(strategy='most_frequent')),  # Changed from constant
                        ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
                    ])
                    transformers.append(('categorical_low', onehot_pipe, low_card_cols))
        
        return ColumnTransformer(transformers, remainder='drop', sparse_threshold=0)
    
    def fit(self, X, y):
        """
        Fit the robust estimator with mixed data types
        """
        # Ensure input is DataFrame
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        # Convert y to numpy array
        y = np.asarray(y)
        
        # Infer column types if not provided
        self._infer_column_types(X)
        
        # Create and fit preprocessing pipeline
        self.preprocessor_ = self._create_preprocessing_pipeline(X)
        X_transformed = self.preprocessor_.fit_transform(X, y)
        
        # Initialize the final estimator
        self.final_estimator_ = RandomForestRegressor(n_estimators=100, random_state=self.random_state)
        self.final_estimator_.fit(X_transformed, y)
        
        # Get actual number of features and importances
        n_features_actual = X_transformed.shape[1]
        feature_importances = self.final_estimator_.feature_importances_

        # Create basic feature names first
        feature_names = [f"feature_{i}" for i in range(n_features_actual)]

        # Ensure we have matching lengths
        if len(feature_importances) != len(feature_names):
            min_len = min(len(feature_importances), len(feature_names))
            feature_importances = feature_importances[:min_len]
            feature_names = feature_names[:min_len]
        
        # Print diagnostic information
        print("\nFeature Importance Summary:")
        print("-" * 50)
        print(f"Total number of features: {len(feature_names)}")
        
        # Create and sort importance dictionary
        importance_dict = dict(zip(feature_names, feature_importances))
        sorted_importances = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)
        
        print("\nTop 10 Most Important Features:")
        for name, importance in sorted_importances[:10]:
            print(f"{name:20} : {importance:.4f}")

        # Store feature importances with matching lengths
        self.feature_names_ = feature_names
        self.feature_importances_ = pd.Series(
            feature_importances,
            index=feature_names
        )
        
        return self
    
    def predict(self, X):
        """
        Make predictions using the fitted model
        """
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
            
        X_transformed = self.preprocessor_.transform(X)
        return self.final_estimator_.predict(X_transformed)
    

    def get_feature_importance_summary(self):
        """
        Get detailed feature importance summary
        """
        if self.feature_importances_ is None:
            return None
                
        importance_df = pd.DataFrame({
            'feature': self.feature_importances_.index,
            'importance': self.feature_importances_.values
        })
        
        return importance_df.sort_values('importance', ascending=False)
    
    def get_feature_mapping(self, X):
        """
        Map transformed feature indices to original columns
        """
        # Get original column names
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        print("\nFeature Mapping:")
        print("-" * 50)
        
        # Numeric columns
        if self.numeric_cols:
            print("\nNumeric columns:")
            for col in self.numeric_cols:
                print(f"Column '{col}' -> likely feature_[i] where i is a single index")
                
        # Boolean columns
        if self.boolean_cols:
            print("\nBoolean columns:")
            for col in self.boolean_cols:
                print(f"Column '{col}' -> likely feature_[i] where i is a single index")
                
        # Categorical columns
        if self.categorical_cols:
            print("\nCategorical columns:")
            for col in self.categorical_cols:
                n_unique = len(X[col].dropna().unique())
                if n_unique <= 10:  # one-hot encoded
                    print(f"Column '{col}' -> likely features_[i] where i spans {n_unique} consecutive indices")
                else:  # mean encoded
                    print(f"Column '{col}' -> likely feature_[i] where i is a single index")
                    
        print("\nNote: Feature indices are assigned sequentially in the order: numeric -> boolean -> categorical")
    
    def identify_interactions(self, X, threshold=0.01):
        """
        Identify potential interaction terms between features
        """
        from sklearn.inspection import partial_dependence
        import itertools
        
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        # Transform features
        X_transformed = self.preprocessor_.transform(X)
        n_features = X_transformed.shape[1]  # Get actual number of features after transformation
        
        # Get important features above threshold
        important_features = self.feature_importances_[self.feature_importances_ > threshold]
        
        if len(important_features) < 2:
            print("Not enough important features found for interaction analysis.")
            return pd.DataFrame()
        
        # Initialize interaction storage
        interactions = []
        
        # Get all possible feature pairs from the transformed feature space
        feature_pairs = list(itertools.combinations(range(n_features), 2))
        
        print(f"Analyzing interactions among {n_features} features...")
        
        for feat1, feat2 in feature_pairs:
            try:
                # Calculate individual partial dependence
                pd_i = partial_dependence(
                    self.final_estimator_, 
                    X_transformed, 
                    [feat1], 
                    grid_resolution=10  # Lower resolution for speed
                )[1][0]
                
                pd_j = partial_dependence(
                    self.final_estimator_, 
                    X_transformed, 
                    [feat2], 
                    grid_resolution=10
                )[1][0]
                
                # Calculate joint partial dependence
                pd_ij = partial_dependence(
                    self.final_estimator_, 
                    X_transformed, 
                    [[feat1, feat2]], 
                    grid_resolution=10
                )[1][0]
                
                # Calculate interaction strength
                interaction_strength = np.abs(pd_ij - np.outer(pd_i, pd_j)).mean()
                
                # Get feature names
                feature1_name = f"feature_{feat1}"
                feature2_name = f"feature_{feat2}"
                
                # Get feature importances if available
                importance1 = self.feature_importances_.get(feature1_name, 0)
                importance2 = self.feature_importances_.get(feature2_name, 0)
                
                interactions.append({
                    'feature1': feature1_name,
                    'feature2': feature2_name,
                    'importance1': importance1,
                    'importance2': importance2,
                    'interaction_strength': interaction_strength,
                    'combined_importance': importance1 * importance2
                })
                
            except Exception as e:
                print(f"Warning: Could not calculate interaction for features {feat1} and {feat2}: {str(e)}")
                continue
        
        if not interactions:
            print("No valid interactions found.")
            return pd.DataFrame()
        
        # Create DataFrame of interactions
        interaction_df = pd.DataFrame(interactions)
        
        # Sort by both interaction strength and combined importance
        interaction_df['overall_score'] = interaction_df['interaction_strength'] * \
                                        interaction_df['combined_importance']
        interaction_df = interaction_df.sort_values('overall_score', ascending=False)
        
        # Print summary
        print("\nTop Feature Interactions:")
        print("-" * 50)
        for _, row in interaction_df.head(5).iterrows():
            print(f"\nInteraction between {row['feature1']} and {row['feature2']}:")
            print(f"Individual Importances: {row['importance1']:.4f}, {row['importance2']:.4f}")
            print(f"Interaction Strength: {row['interaction_strength']:.4f}")
            print(f"Overall Score: {row['overall_score']:.4f}")
        
        return interaction_df

    def get_original_feature_names(self, X):
        """
        Map transformed feature indices back to original column names
        """
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        feature_mapping = {}
        current_idx = 0
        
        # Map numeric features
        for col in self.numeric_cols:
            feature_mapping[f'feature_{current_idx}'] = f'numeric_{col}'
            current_idx += 1
        
        # Map boolean features
        for col in self.boolean_cols:
            feature_mapping[f'feature_{current_idx}'] = f'boolean_{col}'
            current_idx += 1
        
        # Map categorical features
        for col in self.categorical_cols:
            n_unique = len(X[col].dropna().unique())
            if n_unique <= 10:  # one-hot encoded
                for i in range(n_unique):
                    feature_mapping[f'feature_{current_idx + i}'] = f'categorical_{col}_{i}'
                current_idx += n_unique
            else:  # mean encoded
                feature_mapping[f'feature_{current_idx}'] = f'categorical_{col}'
                current_idx += 1
        
        return feature_mapping

def demonstrate_mixed_type_estimation():
    """
    Demonstrate the mixed-type robust estimator
    """
    np.random.seed(42)
    n_samples = 1000
    
    # Create sample dataset
    data = pd.DataFrame({
        'numeric1': np.random.normal(0, 1, n_samples),
        'numeric2': np.random.normal(0, 1, n_samples),
        'boolean1': np.random.choice([0, 1], n_samples),
        'boolean2': np.random.choice([True, False], n_samples),
        'category1': np.random.choice(['A', 'B', 'C'], n_samples),
        'category2': np.random.choice(['low', 'medium', 'high'], n_samples)
    })
    
    # Create target variable
    y = (
        1.5 * data['numeric1'] +
        -0.8 * data['numeric2'] +
        0.5 * data['boolean1'] +
        0.3 * (data['category1'] == 'A').astype(int) +
        np.random.normal(0, 0.1, n_samples)
    )
    
    # Add missing values
    for col in data.columns:
        mask = np.random.random(n_samples) < 0.1
        data.loc[mask, col] = None
    
    # Fit estimator
    estimator = MixedTypeRobustEstimator()
    estimator.fit(data, y)
    
    # Make predictions
    y_pred = estimator.predict(data)
    
    return {
        'r2_score': r2_score(y, y_pred),
        'rmse': np.sqrt(mean_squared_error(y, y_pred)),
        'feature_importance': estimator.get_feature_importance_summary()
    }

In [2]:
def load_data(f_path: str) -> pd.DataFrame:
    """
    Load the dataset and split into features and target
    """
    data = pd.read_csv('data/train.csv', index_col="id")
    X = data.drop('price', axis=1)
    y = data['price']
    return X, y

In [8]:
# Load data
X, y = load_data('data/train.csv')

drop_columns = ["orientation", "neighborhood", "is_furnished", "has_pool", "has_ac", "accepts_pets"]

# Initialize and fit the robust estimator

# Fit estimator
estimator = MixedTypeRobustEstimator(numeric_cols=['num_rooms', 'num_baths', 'square_meters'], boolean_cols=['is_furnished', 'has_pool', 'has_ac', 'accepts_pets'], 
                                     categorical_cols=['orientation', 'year_built', 'door', 'neighborhood', 'num_crimes', 'num_supermarkets'])
estimator.fit(X, y)

# Make predictions
y_pred = estimator.predict(X)

# Get feature importance summary
#importance_summary = estimator.get_feature_importance_summary()

results = {
    'r2_score': r2_score(y, y_pred),
    'mse': mean_squared_error(y, y_pred),
    'rmse': np.sqrt(mean_squared_error(y, y_pred)),
    'feature_importance': estimator.get_feature_importance_summary()
}

print(results)


Feature Importance Summary:
--------------------------------------------------
Total number of features: 26

Top 10 Most Important Features:
feature_2            : 0.5868
feature_7            : 0.1217
feature_0            : 0.0364
feature_1            : 0.0302
feature_3            : 0.0172
feature_5            : 0.0169
feature_6            : 0.0164
feature_4            : 0.0164
feature_12           : 0.0127
feature_8            : 0.0111
{'r2_score': 0.9290155144990805, 'mse': np.float64(5238.9667395915985), 'rmse': np.float64(72.38070695697576), 'feature_importance':        feature  importance
2    feature_2    0.586771
7    feature_7    0.121690
0    feature_0    0.036378
1    feature_1    0.030214
3    feature_3    0.017165
5    feature_5    0.016932
6    feature_6    0.016426
4    feature_4    0.016374
12  feature_12    0.012669
8    feature_8    0.011137
9    feature_9    0.011006
10  feature_10    0.010921
21  feature_21    0.010891
20  feature_20    0.010598
18  feature_18    0.

In [9]:
# Hannes approach
# Load data
X, y = load_data('data/train.csv')

drop_columns = ["orientation", "has_pool", "has_ac", "accepts_pets"]

X = X.drop(drop_columns, axis=1)

# Initialize and fit the robust estimator

# Fit estimator
estimator = MixedTypeRobustEstimator(numeric_cols=['num_rooms', 'num_baths', 'square_meters'],  
                                     categorical_cols=['year_built', 'door', 'num_crimes', 'num_supermarkets', "neighborhood"], boolean_cols=['is_furnished'])
estimator.fit(X, y)

# Make predictions
y_pred = estimator.predict(X)

# Get feature importance summary
#importance_summary = estimator.get_feature_importance_summary()

results = {
    'r2_score': r2_score(y, y_pred),
    'mse': mean_squared_error(y, y_pred),
    'rmse': np.sqrt(mean_squared_error(y, y_pred)),
    'feature_importance': estimator.get_feature_importance_summary()
}

print(results)


Feature Importance Summary:
--------------------------------------------------
Total number of features: 18

Top 10 Most Important Features:
feature_2            : 0.6065
feature_4            : 0.1573
feature_0            : 0.0452
feature_1            : 0.0390
feature_3            : 0.0226
feature_16           : 0.0127
feature_15           : 0.0120
feature_8            : 0.0115
feature_10           : 0.0114
feature_13           : 0.0112
{'r2_score': 0.9268571545278854, 'mse': np.float64(5398.263183333665), 'rmse': np.float64(73.47287379253424), 'feature_importance':        feature  importance
2    feature_2    0.606537
4    feature_4    0.157304
0    feature_0    0.045195
1    feature_1    0.039043
3    feature_3    0.022617
16  feature_16    0.012718
15  feature_15    0.012008
8    feature_8    0.011453
10  feature_10    0.011449
13  feature_13    0.011203
14  feature_14    0.010881
9    feature_9    0.010604
17  feature_17    0.010437
11  feature_11    0.010306
12  feature_12    0.0

In [4]:
estimator.get_feature_mapping(X)


Feature Mapping:
--------------------------------------------------

Numeric columns:
Column 'num_rooms' -> likely feature_[i] where i is a single index
Column 'num_baths' -> likely feature_[i] where i is a single index
Column 'square_meters' -> likely feature_[i] where i is a single index

Categorical columns:
Column 'year_built' -> likely feature_[i] where i is a single index
Column 'door' -> likely feature_[i] where i is a single index
Column 'num_crimes' -> likely feature_[i] where i is a single index
Column 'num_supermarkets' -> likely features_[i] where i spans 3 consecutive indices

Note: Feature indices are assigned sequentially in the order: numeric -> boolean -> categorical


In [5]:
# Get interactions
interactions = estimator.identify_interactions(X, threshold=0.01)

# Get feature mapping
feature_mapping = estimator.get_original_feature_names(X)

# Map feature names in interactions
interactions['feature1_original'] = interactions['feature1'].map(feature_mapping)
interactions['feature2_original'] = interactions['feature2'].map(feature_mapping)

print("\nMapped Feature Interactions:")
print(interactions[['feature1_original', 'feature2_original', 'interaction_strength', 'overall_score']].head())

Analyzing interactions among 7 features...
No valid interactions found.


KeyError: 'feature1'

In [6]:
# Apply to test data
test_data = pd.read_csv('data/test.csv', index_col="id")
test_data = test_data.drop(drop_columns, axis=1)
test_pred = estimator.predict(test_data)
new_pred = pd.DataFrame(test_pred, index=test_data.index, columns=['price'])

In [7]:
new_pred.to_csv('test_predictions.csv', index=True, header=['price'])